In [ ]:
!git clone https://github.com/LuckyBoy587/ComputerVision.git

In [ ]:
import os
import shutil
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models
from tensorflow.keras import mixed_precision

mixed_precision.set_global_policy('mixed_float16')

In [ ]:
if os.path.exists('ComputerVision/data/brain_tumor_dataset'):
    if not os.path.exists('/content/brain_tumor_dataset'):
        shutil.copytree('ComputerVision/data/brain_tumor_dataset', '/content/brain_tumor_dataset')

dataset_path = '/content/brain_tumor_dataset'
folders = sorted(os.listdir(dataset_path))
no_images = len(os.listdir(os.path.join(dataset_path, 'no')))
yes_images = len(os.listdir(os.path.join(dataset_path, 'yes')))
total_samples = no_images + yes_images

print(f"Dataset path: {dataset_path}")
print(f"Folders: {folders}")

datagen = ImageDataGenerator(rescale=1./255)
generator = datagen.flow_from_directory(
    dataset_path,
    target_size=(224, 224),
    batch_size=32,
    class_mode='binary'
)

print(f"Number of samples: {total_samples}")

In [ ]:
base_model = MobileNetV2(input_shape=(224, 224, 3), include_top=False, weights='imagenet')
base_model.trainable = False

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(1, activation='sigmoid', dtype='float32')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

print(f'Model: "{model.name}"')
model.summary()